# Bedrock Module · Day 11 — AWS Bedrock Foundations

**Phase 1 · Module 2: AWS Bedrock & AgentCore** · ~2.5 hrs

Module 1 you built agents with LangGraph. But an agent is only as good as the **model** behind it — and at Barclays that model is served by **Amazon Bedrock**: one managed API that fronts many foundation models (Claude, Nova, Titan, Mistral) with the bank's security, logging, and guardrails already wired in. Today is the on-ramp: **what Bedrock is, which models it offers, how to run inference two ways, and how to read the token bill.**

> **Keyless & offline.** Real Bedrock needs an AWS account, enabled model access, and IAM. So this notebook uses the **same `FakeBedrockRuntime`** shape as this morning's Python lesson — identical method names and response shapes to the real `boto3` client. Everything you write here is real boto3 code; only the client is swapped. Promote it to production by changing **one line** (§8 shows exactly which).


## 0. Setup — a keyless stand-in for `boto3`

In a real environment step 0 is: create an AWS account → open the Bedrock console → **request model access** (Claude/Nova/etc. are off by default) → create IAM keys → `pip install boto3`. Then:

```python
import boto3
brt = boto3.client("bedrock-runtime", region_name="eu-west-2")   # London region
```

We reproduce that client offline so the lesson runs anywhere.


In [1]:
import json

class ClientError(Exception):
    def __init__(self, code, message):
        self.response = {"Error": {"Code": code, "Message": message}}
        super().__init__(f"{code}: {message}")

# canned 'model personalities' so different model ids give visibly different replies
_STYLE = {
    "claude": "careful, cites policy",
    "nova":   "fast, terse",
    "titan":  "plain, conservative",
    "mistral":"open-weight, direct",
}

class FakeBedrockRuntime:
    def _family(self, mid):
        for fam in _STYLE:
            if fam in mid: return fam
        return None
    def converse(self, modelId, messages, inferenceConfig=None, **kw):
        fam = self._family(modelId)
        if fam is None:
            raise ClientError("ValidationException", f"model '{modelId}' is not enabled in this account")
        q = messages[-1]["content"][0]["text"]
        reply = f"({_STYLE[fam]}) {fam.title()} answer to: {q[:50]}"
        it, ot = max(1, len(q.split())), max(1, len(reply.split()))
        return {"output": {"message": {"role": "assistant", "content": [{"text": reply}]}},
                "stopReason": "end_turn",
                "usage": {"inputTokens": it, "outputTokens": ot, "totalTokens": it + ot}}
    def converse_stream(self, modelId, messages, **kw):
        full = self.converse(modelId, messages, **kw)
        words = full["output"]["message"]["content"][0]["text"].split()
        # AWS streams a sequence of event dicts; we yield the deltas that matter
        for w in words:
            yield {"contentBlockDelta": {"delta": {"text": w + " "}}}
        yield {"metadata": {"usage": full["usage"]}}
    def invoke_model(self, modelId, body, **kw):
        p = json.loads(body)
        txt = p.get("inputText") or p.get("messages", [{}])[-1].get("content", "")
        out = json.dumps({"results": [{"outputText": f"titan echo: {txt[:40]}"}]}).encode()
        return {"body": _Body(out)}

class _Body:
    def __init__(self, d): self._d = d
    def read(self): return self._d

brt = FakeBedrockRuntime()
print("bedrock-runtime client ready (offline stand-in)")


bedrock-runtime client ready (offline stand-in)


## 1. The model catalogue

Bedrock's pitch: **one API, many models.** You pick a model by its **model id** and the rest of your code is identical. These are the families you'll be asked about in interviews and the kind of job each is picked for at a bank:

| Family | Example model id | Sweet spot |
|---|---|---|
| **Anthropic Claude** | `eu.anthropic.claude-sonnet-4-5-20250929-v1:0` | reasoning, tool use, long context — the agent default |
| **Amazon Nova** | `eu.amazon.nova-pro-v1:0` | fast, cheap, high-volume classification |
| **Amazon Titan** | `amazon.titan-text-express-v1` | simple text + **embeddings** (`titan-embed`) |
| **Mistral** | `mistral.mistral-large-2407-v1:0` | open-weight, EU-hosted option |

Note the `eu.` prefix on some ids — that's a **cross-region inference profile**, which keeps traffic inside the EU (data-residency matters a lot for a UK bank). Let's register the ones we'll use.


In [2]:
CATALOGUE = {
    "claude":  "eu.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "nova":    "eu.amazon.nova-pro-v1:0",
    "titan":   "amazon.titan-text-express-v1",
    "mistral": "mistral.mistral-large-2407-v1:0",
}
for name, mid in CATALOGUE.items():
    print(f"{name:8} -> {mid}")


claude   -> eu.anthropic.claude-sonnet-4-5-20250929-v1:0
nova     -> eu.amazon.nova-pro-v1:0
titan    -> amazon.titan-text-express-v1
mistral  -> mistral.mistral-large-2407-v1:0


## 2. Run inference with the **Converse** API

Converse is the unified, model-agnostic way to chat. Same `messages` shape you learned this morning: a list of `{role, content:[{text}]}`. Because the shape is identical across models, **swapping the model id is the only change** needed to try a different brain on the same task.


In [3]:
def ask(model_id, prompt, **cfg):
    resp = brt.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 300, "temperature": 0.0, **cfg},
    )
    text = resp["output"]["message"]["content"][0]["text"]
    return text, resp["usage"]

prompt = "A customer disputes a 89.99 GBP card charge. Draft a one-line next action."
text, usage = ask(CATALOGUE["claude"], prompt)
print(text)
print("usage:", usage)


(careful, cites policy) Claude answer to: A customer disputes a 89.99 GBP card charge. Draft
usage: {'inputTokens': 13, 'outputTokens': 15, 'totalTokens': 28}


☝ One helper, any model. The reply text is at the same path every time and `usage` rides along on every call. Keep `temperature=0.0` for bank work — you want the **same input to give the same output** so decisions are auditable and reproducible.


## 3. Same task, four models — the value of one API

Because only the id changes, you can **shop around**: send the identical prompt to every family and compare answer *and* token cost. This is how teams decide 'Nova for the cheap high-volume step, Claude for the hard reasoning step.'


In [4]:
task = "Classify this spend in one word: 'AWS EU compute invoice 420.50 GBP'."
for name, mid in CATALOGUE.items():
    text, usage = ask(mid, task)
    print(f"{name:8} | {usage['totalTokens']:>3} tok | {text}")


claude   |  28 tok | (careful, cites policy) Claude answer to: Classify this spend in one word: 'AWS EU compute i
nova     |  27 tok | (fast, terse) Nova answer to: Classify this spend in one word: 'AWS EU compute i
titan    |  27 tok | (plain, conservative) Titan answer to: Classify this spend in one word: 'AWS EU compute i
mistral  |  27 tok | (open-weight, direct) Mistral answer to: Classify this spend in one word: 'AWS EU compute i


☝ Four different models, **one line of calling code**, driven by a dict lookup. That portability is Bedrock's core promise: your agent isn't married to a vendor — a config change re-points it. (In a real account each of these must be individually **access-enabled** first, or you'd get the `ValidationException` from §6.)


## 4. Streaming with **ConverseStream**

For anything a human waits on, stream the answer token-by-token so the UI feels instant (you did the Python side on Day 07). `converse_stream` returns a sequence of **event** dicts; you pull text out of `contentBlockDelta` events and the final token count out of the `metadata` event.


In [5]:
print("streaming: ", end="")
final_usage = None
for event in brt.converse_stream(
        modelId=CATALOGUE["claude"],
        messages=[{"role": "user", "content": [{"text": "Summarise KYC in 6 words"}]}]):
    if "contentBlockDelta" in event:
        print(event["contentBlockDelta"]["delta"]["text"], end="", flush=True)
    elif "metadata" in event:
        final_usage = event["metadata"]["usage"]
print("\n[done] usage:", final_usage)


streaming: (careful, cites policy) Claude answer to: Summarise KYC in 6 words 
[done] usage: {'inputTokens': 5, 'outputTokens': 11, 'totalTokens': 16}


☝ Two event types matter: `contentBlockDelta` (a chunk of text) and `metadata` (the final bill). Same nested-dict discipline as the non-streaming call — you just consume it as it arrives instead of all at once.


## 5. The older **InvokeModel** API (per-model body)

Some models/features predate Converse and need `invoke_model` with a **model-specific** JSON body. Titan text, for instance, wants `{"inputText": ...}` and returns `results[0]["outputText"]`. Recognise the pattern — you'll meet it again for **embeddings** in the Knowledge Bases lesson (Day 12).


In [6]:
body = json.dumps({"inputText": "Normalise vendor name: 'AMZN AWS EMEA'",
                   "textGenerationConfig": {"maxTokenCount": 100, "temperature": 0.0}})

raw = brt.invoke_model(modelId=CATALOGUE["titan"], body=body)
parsed = json.loads(raw["body"].read())
print(parsed["results"][0]["outputText"])


titan echo: Normalise vendor name: 'AMZN AWS EMEA'


☝ Different envelope, same idea: `json.dumps` in, `body.read()` + `json.loads` out. Default to **Converse**; drop to `invoke_model` only when a model or feature needs it.


## 6. Token limits & error handling

Every model has a **context window** (max input+output tokens) and per-account **rate limits**. Exceed either and Bedrock raises a `ClientError` — the same code-branching you learned this morning. The two you'll hit most: `ValidationException` (model not enabled / request too big) and `ThrottlingException` (slow down).


In [7]:
def robust_ask(model_id, prompt):
    try:
        return ask(model_id, prompt)[0]
    except ClientError as e:
        code = e.response["Error"]["Code"]
        return f"[handled {code}] {e.response['Error']['Message']}"

print(robust_ask(CATALOGUE["nova"], "ping"))               # ok
print(robust_ask("cohere.command-r-v1:0", "ping"))         # not enabled -> ValidationException


(fast, terse) Nova answer to: ping
[handled ValidationException] model 'cohere.command-r-v1:0' is not enabled in this account


☝ The enabled model answers; the un-enabled one is **caught and explained** instead of crashing the agent. In a real graph this is the difference between one bad node failing gracefully and the whole pipeline going down.


## 7. Pricing — turn `usage` into pounds

Bedrock bills **per 1,000 tokens**, priced separately for input and output, per model. You already capture `usage` on every call — so cost is just arithmetic. Wire this into your logging and you get a live spend meter for free. (Rates below are **illustrative** — always check the current Bedrock pricing page.)


In [8]:
# illustrative GBP per 1K tokens (input, output)
PRICE = {
    "claude":  (0.0024, 0.012),
    "nova":    (0.0006, 0.0024),
    "titan":   (0.0002, 0.0006),
    "mistral": (0.0016, 0.0048),
}

def cost_gbp(name, usage):
    pin, pout = PRICE[name]
    return usage["inputTokens"]/1000*pin + usage["outputTokens"]/1000*pout

task = "Draft a 2-sentence dispute acknowledgement email."
print(f"{'model':8} {'tokens':>7} {'cost (GBP)':>12}")
for name, mid in CATALOGUE.items():
    _, usage = ask(mid, task)
    print(f"{name:8} {usage['totalTokens']:>7} {cost_gbp(name, usage):>12.6f}")


model     tokens   cost (GBP)
claude        18     0.000158
nova          17     0.000030
titan         17     0.000008
mistral       17     0.000062


☝ Same task, and now you can *see* why model choice is a budget decision: at scale (millions of invoices) the gap between Nova and Claude is real money. The rule at a bank: **cheapest model that clears the accuracy bar for that step** — which is exactly what a multi-model agent lets you do.


## 8. From fake to real — the one-line switch

Everything above is production boto3 code. To run it for real:

```python
import boto3
brt = boto3.client("bedrock-runtime", region_name="eu-west-2")   # <-- replace FakeBedrockRuntime()
from botocore.exceptions import ClientError                       # <-- replace our ClientError
```

…plus, in the AWS console: enable model access for each id in `CATALOGUE`, and give your IAM role `bedrock:InvokeModel` / `bedrock:InvokeModelWithResponseStream`. **Your `ask`, `robust_ask`, and `cost_gbp` functions do not change** — that's the reward for coding against the API shape.


## 9. Recap — you can drive Bedrock

| Piece | You learned | Reused from |
|---|---|---|
| **client** | `boto3.client("bedrock-runtime", region_name=...)` | Python Day 11 |
| **catalogue** | pick a model by **id**; `eu.` = EU data residency | today |
| **Converse** | unified `messages` shape, swap id to swap model | Python Day 11 |
| **stream** | `converse_stream` → `contentBlockDelta` + `metadata` | LangGraph Day 07 |
| **InvokeModel** | per-model body, `read()` bytes back | Python Day 11 §4 |
| **errors** | branch on `ClientError` `Error.Code` | Python Day 05 |
| **cost** | `usage` × per-1K price = live spend meter | today |

